In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# from kret_studies import *
# from kret_studies.notebook import *
# from kret_studies.complex import *

# logger = get_notebook_logger()

In [3]:
from waymo_agent.notebook_imports import *

In [4]:
from functools import cache

In [5]:
from waymo_agent import *
from waymo_agent.osmnx import *
from waymo_agent.data_classes import *
from waymo_agent.action_heuristic import *
from waymo_agent.graph_env import *
from waymo_agent.simulation import *
from waymo_agent.action_heuristic.heuristic_simple import PricingAgent, DispatchAgent, RepositionAgent
from waymo_agent.models.ppo_model import RideShareActorCritic

In [6]:
from kret_sandbox.VIS import dtt
from kret_sandbox.exp_decay import exp_decay_half_life, get_gamma_from_half_life

In [7]:
def get_obs_tuple(env: RideShareEnv):
    veh = env.observation_curr["vehicles"]
    req = env.observation_curr["pending_requests"]
    rides = env.observation_curr["active_rides"]
    return veh, req, rides


def get_obs_tuple_from_obs(obs: ObservationDict):
    veh = obs["vehicles"]
    req = obs["pending_requests"]
    rides = obs["active_rides"]
    return veh, req, rides

# Run Sim

In [8]:
config = EnvConfig(max_episode_steps=60 * 3)
plt_cfg = PlotConfig()

In [9]:
GAMMA = get_gamma_from_half_life(config.max_episode_steps // 2)
round(GAMMA, 5)

0.99233

In [10]:
def discounted_rewards(rewards: np.ndarray, gamma: float = 0.997):
    """
    Discount rewards using discount factor gamma.
    """
    disc_schedule = exp_decay_half_life(len(rewards), gamma=gamma)
    return rewards * disc_schedule


def get_act_dict(agents: tuple[PricingAgent, DispatchAgent, RepositionAgent], obs: ObservationDict) -> ActionDict:
    price_agent, dispatch_agent, reposition_agent = agents
    prices = price_agent.price(obs)
    dispatch_actions = dispatch_agent.dispatch(obs)
    reposition_actions = reposition_agent.reposition(obs)

    action_agent: ActionDict = {
        "prices": prices,
        "dispatch": dispatch_actions,
        "reposition": reposition_actions,
    }
    return action_agent

In [11]:
def run_heuristic_simulation(env_curr: RideShareEnv, num_iter: int = 1):
    REWARDS: list[list[float]] = []
    OBS: list[list[ObservationDict]] = []
    ACT: list[list[ActionDict]] = []

    agents = (PricingAgent(env_curr.config), DispatchAgent(env_curr), RepositionAgent(env_curr))

    tqdm.write("Starting heuristic simulation...")

    for iter_count in tqdm(range(num_iter)):
        rews_raw: list[float] = []
        obs_list: list[ObservationDict] = []
        act_list: list[ActionDict] = []
        done = False
        obs, info = env_curr.reset()

        while not done:
            obs: ObservationDict
            obs_list.append(obs)

            action_agent: ActionDict = get_act_dict(agents, obs)
            act_list.append(action_agent)

            obs, reward, term, done, info = env_curr.step(action_agent)  # type: ignore

            rews_raw.append(reward)
            if term:
                tqdm.write(f"WARNING: Episode terminated at step {env_curr.current_step}")
            done = done or term

        rews = discounted_rewards(np.array(rews_raw), gamma=GAMMA).tolist()
        REWARDS.append(rews)
        OBS.append(obs_list)
        ACT.append(act_list)
    return (REWARDS), OBS, ACT, env_curr, agents

In [12]:
env_heuristic = RideShareEnv(config, plt_cfg)
G = env_heuristic.G
cfg = env_heuristic.config

Assigned lambda values to nodes. Total lambda: 2.2210 (target: 2.3460)


In [13]:
REWARDS, OBS, ACT, env_curr, agents = run_heuristic_simulation(env_heuristic, num_iter=1)

Starting heuristic simulation...


100%|██████████| 1/1 [00:11<00:00, 11.73s/it]


In [14]:
arr = np.array(REWARDS)
arr.shape

(1, 181)

In [15]:
arr2 = arr.reshape(-1, arr.shape[0])
arr2.shape

(181, 1)

In [16]:
pd.DataFrame(arr2).head(2)

,0
0,0.000000
1,0.297698


## NN Actor-Critic

In [17]:
from waymo_agent.graph_env.ENV import RideShareEnv
from waymo_agent.models.ppo_model import (
    train_ppo,
    PPOTrainConfig,
    RideShareActorCritic,
    obs_pd_to_torch,
    action_torch_to_numpy,
)

In [25]:
config = EnvConfig(max_episode_steps=60 * 3)
plt_cfg = PlotConfig()
env_model = RideShareEnv(config, plt_cfg)

Assigned lambda values to nodes. Total lambda: 2.2821 (target: 2.3460)


In [26]:
cfg_ppo = PPOTrainConfig()
model = RideShareActorCritic(env_curr)

In [27]:
@torch.no_grad()
def run_model_simulation(
    env_curr: RideShareEnv,
    model: RideShareActorCritic,
    num_iter: int = 1,
    gamma: float = GAMMA,
    deterministic: bool = False,
):
    """
    Returns:
      REWARDS: list[episode][t] discounted reward_t
      OBS:     list[episode][t] obs dict (numpy)
      ACT:     list[episode][t] action dict (numpy)
    """
    model.eval()

    REWARDS: list[list[float]] = []
    OBS: list[list[dict[str, np.ndarray]]] = []
    ACT: list[list[dict[str, np.ndarray]]] = []

    for i in tqdm(range(num_iter), desc="model rollout"):
        print(f"Starting episode {i+1}/{num_iter}...")
        obs_np, _info = env_curr.reset()
        done = False

        rews_raw: list[float] = []
        obs_list: list[dict[str, np.ndarray]] = []
        act_list: list[dict[str, np.ndarray]] = []

        while not done:
            print(f"Step {env_curr.current_step}/{env_curr.config.max_episode_steps}")
            obs_list.append(obs_np)

            obs_t = obs_pd_to_torch(obs_np)

            act_t = model.act(obs_t, deterministic=deterministic)
            act_np = action_torch_to_numpy(act_t)
            env_curr._validate_action(act_np)
            act_list.append(act_np)

            obs_np, reward, terminated, truncated, _info = env_curr.step(act_np)  # type: ignore[arg-type]
            rews_raw.append(float(reward))
            done = bool(terminated) or bool(truncated)

        rews = discounted_rewards(np.array(rews_raw, dtype=np.float32), gamma=gamma).tolist()
        REWARDS.append(t.cast(list[float], rews))
        OBS.append(obs_list)
        ACT.append(act_list)

    return REWARDS, OBS, ACT

In [28]:
out = run_model_simulation(env_model, model, num_iter=1)

model rollout:   0%|          | 0/1 [00:00<?, ?it/s]

Starting episode 1/1...
Step 0/180


TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [36]:
obs, info = env_curr.reset()
obs_t = obs_pd_to_torch(obs)

act_t = model.act(obs_t, deterministic=True)

ValueError: Failed to convert obs key 'pending_requests' with type object to tensor.

In [37]:
obs["pending_requests"].dtypes

pickup_x_norm               float64
pickup_y_norm               float64
dropoff_x_norm              float64
dropoff_y_norm              float64
distance_meters             float64
cust_bias                   float64
cust_temperature            float64
est_cost                    float64
max_wait_time       timedelta64[ns]
wait_time            timedelta64[s]
status                        int64
dtype: object

In [ ]:
model, logs = ppo_model.train_ppo(env_model, model, cfg=cfg_ppo)

AttributeError: 'EnvConfig' object has no attribute 'lr'

In [ ]:
# env.breadcrumbs["rewards"].cumsum().plot(title="Cumulative Reward over Time")
# plt.xlabel("Time Step")
# plt.ylabel("Cumulative Reward")
# plt.show()

In [ ]:
# fig, ax = env.render()
# fig